# Train Point-ACT on Kaggle (2x T4) — pencil pickup -> mouse pad

Trains the **Point-ACT** policy (point-conditioned WM-ACT, ~53M params) on the
`fouad1233/so101_pencil_pickup` dataset and pushes it to the Hub.

**Before running:**
1. Kaggle Settings -> Accelerator **GPU T4 x2**, **Internet ON**.
2. Add-ons -> Secrets -> `HF_TOKEN` (huggingface.co/settings/tokens, *write* scope).
3. On your Mac, push the branch with the plugin code: `git push -u origin wm-act`
   (this notebook clones `wm-act` from github.com/fouad1233/fouad_so101).


## 1. GPUs


In [ ]:
!nvidia-smi


## 2. Install LeRobot + clone the plugin code
LeRobot 0.5.2 is not on PyPI — installed from git, **pinned to the exact commit**
the repo's submodule uses, so the API matches what `point_act`/`wm_act` were
written against. The clone skips the lerobot submodule (the pip package is used).


In [ ]:
LEROBOT_COMMIT = "6a788fbdb02cabfae60f7408636945df0b1eafa0"  # = repo submodule pin
!pip install -q "lerobot[dataset] @ git+https://github.com/huggingface/lerobot.git@{LEROBOT_COMMIT}" transformers accelerate
!git clone --depth 1 --branch wm-act https://github.com/fouad1233/fouad_so101.git /kaggle/working/fouad_so101

import os, sys
REPO = "/kaggle/working/fouad_so101"
sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")


## 3. Hugging Face login (from the `HF_TOKEN` Kaggle secret)


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))


## 4. Sanity check before burning GPU time (~1 min)


In [ ]:
!cd /kaggle/working/fouad_so101 && PYTHONPATH=. python point_act/smoke_test.py


## 5. Settings — edit these


In [ ]:
HF_USER      = "fouad1233"
DATASET_REPO = f"{HF_USER}/so101_pencil_pickup"
MODEL_REPO   = f"{HF_USER}/pointact_pencil_pickup"

NUM_GPUS   = 2      # T4 x2 -> 2; set 1 for a single GPU
# BATCH_SIZE is PER-GPU (effective batch = BATCH_SIZE x NUM_GPUS). Point-ACT loads
# TWO 640x480 frames per sample (current + future, for the world-model loss), so
# it is heavier per sample than plain ACT. 12 fits a 16GB T4 comfortably.
BATCH_SIZE = 12
STEPS      = 40000  # ~22 epochs at effective batch 24; ~3-4h on 2x T4
SAVE_FREQ  = 5000   # checkpoints for --resume if the session dies
OUTPUT_DIR = "/kaggle/working/train_pointact"
LABELS     = "/kaggle/working/point_labels.json"


## 6. Auto-label the dataset with OWLv2 — tracked every 10 frames (~15 min on GPU)
Locates the pen (pick point) and the mouse pad (place point) **every 10 frames
through each episode**, not just frame 0 — so the green marker FOLLOWS the pen
while it is grasped and carried. The policy learns marker-relative behavior with
markers that stay truthful mid-motion; at robot runtime `run.sh --reground-every`
does the same re-location while moving. OWLv2 runs on the GPU automatically.


In [ ]:
!cd /kaggle/working/fouad_so101 && PYTHONPATH=. python so_brain/relabel.py \
    --repo-id {DATASET_REPO} --pick "a pen" --place "a black mouse pad" \
    --every 10 --out {LABELS}

import json
labels = json.load(open(LABELS))
print(f"{len(labels)}/50 episodes labeled")
assert len(labels) >= 45, "too many detection failures - inspect before training"


## 7. Train (both GPUs via accelerate)
If the session dies, re-run this cell with `"--resume=true"` appended and
`--config_path={OUTPUT_DIR}/checkpoints/last/pretrained_model/train_config.json`.


In [ ]:
import os, shutil, subprocess

train_args = [
    "--policy.discover_packages_path=point_act",
    "--policy.type=point_act",
    f"--policy.point_labels_path={LABELS}",
    f"--dataset.repo_id={DATASET_REPO}",
    "--dataset.video_backend=pyav",
    f"--batch_size={BATCH_SIZE}",
    f"--steps={STEPS}",
    f"--save_freq={SAVE_FREQ}",
    f"--output_dir={OUTPUT_DIR}",
    "--job_name=pointact_pencil",
    "--policy.device=cuda",
    "--policy.push_to_hub=false",
    "--wandb.enable=false",
]

if NUM_GPUS > 1:
    cmd = ["accelerate", "launch", "--multi_gpu", f"--num_processes={NUM_GPUS}",
           shutil.which("lerobot-train"), *train_args]
else:
    cmd = ["lerobot-train", *train_args]

env = dict(os.environ, PYTHONPATH="/kaggle/working/fouad_so101")
print(" ".join(cmd))
subprocess.run(cmd, env=env, check=True)


## 8. Push the trained policy to the Hub


In [ ]:
from huggingface_hub import HfApi
from pathlib import Path

ckpt = Path(OUTPUT_DIR) / "checkpoints" / "last" / "pretrained_model"
assert ckpt.exists(), f"checkpoint not found at {ckpt}"
api = HfApi()
api.create_repo(MODEL_REPO, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=str(ckpt), repo_id=MODEL_REPO, repo_type="model")
print("Pushed to https://huggingface.co/" + MODEL_REPO)


## Done
Back on your Mac (robot + camera connected):
```bash
./so_brain/run.sh "put the pen on the black mouse pad" --model fouad1233/pointact_pencil_pickup
```
No `--rename_map` needed — the policy is trained natively on the `front` camera.
Inference is a sync 30Hz loop on MPS (no RTC needed at 53M params).
